<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Sentiment%20Analysis%20on%20Movie%20Reviews%20using%20Multiple%20Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis on Movie Reviews using Multiple Models

## Introduction

This notebook demonstrates the process of performing sentiment analysis on a movie review dataset. We will compare the effectiveness of different models and embeddings in classifying sentiments (positive vs. negative).


## Data Preprocessing and Pipelines

### Import Libraries

In [42]:
import warnings
warnings.filterwarnings('ignore')

import os
# Check if we are running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

base_path = None
if IN_COLAB:
    # Mount Google Drive
    drive.mount('/content/drive')
    # Define base path
    base_path = '/content/drive/MyDrive'
else:
    # Define base path relative to the local directory where the notebook is running
    base_path = 'content/drive/MyDrive'
    # Create the necessary directory structure if it doesn't exist
    os.makedirs(base_path, exist_ok=True)

In [43]:
# Import necessary libraries
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from gensim.models import KeyedVectors
import gensim.downloader as api

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /home/sreent/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/sreent/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Load and Clean Dataset

In [44]:
# Load and process your dataset with an option to use a fraction of the data
def load_and_prepare_data(file_path, fraction=1.0):
    data = pd.read_csv(file_path)
    if fraction < 1.0:
        data = data.sample(frac=fraction, random_state=42)

    return data

# Load dataset
URL = "https://drive.google.com/file/d/17zquac-Q4viIEs1hSwHmlBIFYXncn9IB/view?usp=sharing"
file_path = "https://drive.google.com/uc?export=download&id=" + URL.split("/")[-2]
data = load_and_prepare_data(file_path, fraction=0.10)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Function to clean text for Bag-of-Words and TF-IDF
def clean_text_bow_tfidf(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])
    # Lemmatize words and remove stopwords
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stopwords.words('english')])
    return text

# Function to clean text for embeddings
def clean_text_embedding(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])
    return text

# Apply the cleaning functions
data['cleaned_text_bow_tfidf'] = data['text'].apply(clean_text_bow_tfidf)
data['cleaned_text_embedding'] = data['text'].apply(clean_text_embedding)

# Check if the dataset is balanced
class_counts = data['tag'].value_counts()
print("Class distribution:\n", class_counts)
print("Class distribution percentage:\n", class_counts / len(data))

# Split the data into training and testing sets for both types of preprocessing
X_train_bow_tfidf, X_test_bow_tfidf, y_train_bow_tfidf, y_test_bow_tfidf = train_test_split(data['cleaned_text_bow_tfidf'], data['tag'], test_size=0.2, random_state=42)
X_train_embedding, X_test_embedding, y_train_embedding, y_test_embedding = train_test_split(data['cleaned_text_embedding'], data['tag'], test_size=0.2, random_state=42)

# Tokenize the text for embedding-based models
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train_embedding)
X_train_seq = tokenizer.texts_to_sequences(X_train_embedding)
X_test_seq = tokenizer.texts_to_sequences(X_test_embedding)

max_seq_length = 200  # Set the desired sequence length

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=max_seq_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_seq_length)

# Convert labels to binary for embedding models
y_train = y_train_embedding.map({'pos': 1, 'neg': 0}).values
y_test = y_test_embedding.map({'pos': 1, 'neg': 0}).values


Class distribution:
 tag
pos    3274
neg    3198
Name: count, dtype: int64
Class distribution percentage:
 tag
pos    0.505871
neg    0.494129
Name: count, dtype: float64


## Bag of Words and TF-IDF

### Naive Bayes with Bag of Words

In [45]:
# Create a pipeline for Naive Bayes with Bag of Words
pipeline_bow_nb = Pipeline([
    ('vectorizer', CountVectorizer(max_features=5000)),
    ('classifier', MultinomialNB())
])

# Perform hyper-parameter optimization with GridSearchCV
param_grid_nb = {
    'classifier__alpha': [0.0, 0.1, 0.5, 1.0]
}

#  Perform grid search with cross-validation
grid_bow_nb = GridSearchCV(pipeline_bow_nb, param_grid=param_grid_nb, cv=3, scoring='accuracy')
grid_bow_nb.fit(X_train_bow_tfidf, y_train_bow_tfidf)

# Best hyper-parameters and cross-validation score
print("Best Hyper-parameters for Naive Bayes with Bag of Words:", grid_bow_nb.best_params_)
print("Best Cross-validation Accuracy for Naive Bayes with Bag of Words:", grid_bow_nb.best_score_)

# Evaluate the best model on the test set
y_pred_bow_nb = grid_bow_nb.predict(X_test_bow_tfidf)
nb_bow_test_accuracy = accuracy_score(y_test_bow_tfidf, y_pred_bow_nb)
print("Test Accuracy:", nb_bow_test_accuracy)

Best Hyper-parameters for Naive Bayes with Bag of Words: {'classifier__alpha': 1.0}
Best Cross-validation Accuracy for Naive Bayes with Bag of Words: 0.5895316304767663
Test Accuracy: 0.5861003861003861


### Naive Bayes with TF-IDF

In [46]:
# Create a pipeline for Naive Bayes with TF-IDF
pipeline_tfidf_nb = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=5000)),
    ('classifier', MultinomialNB())
])

# Perform hyper-parameter optimization with GridSearchCV
param_grid_nb = {
    'classifier__alpha': [0.1, 0.5, 1.0, 1.5, 2.0]
}

# Perform grid search with cross-validation
grid_tfidf_nb = GridSearchCV(pipeline_tfidf_nb, param_grid=param_grid_nb, cv=3, scoring='accuracy')
grid_tfidf_nb.fit(X_train_bow_tfidf, y_train_bow_tfidf)

# Best hyper-parameters and cross-validation score
print("Best Hyper-parameters for Naive Bayes with TF-IDF:", grid_tfidf_nb.best_params_)
print("Best Cross-validation Accuracy for Naive Bayes with TF-IDF:", grid_tfidf_nb.best_score_)

# Evaluate the best model on the test set
y_pred_tfidf_nb = grid_tfidf_nb.predict(X_test_bow_tfidf)
nb_tfidf_test_accuracy = accuracy_score(y_test_bow_tfidf, y_pred_tfidf_nb)
print("Test Accuracy:", nb_tfidf_test_accuracy)

Best Hyper-parameters for Naive Bayes with TF-IDF: {'classifier__alpha': 2.0}
Best Cross-validation Accuracy for Naive Bayes with TF-IDF: 0.5914635498009976
Test Accuracy: 0.5891891891891892


### Logistic Regression with Bag of Words

In [47]:
# Create a pipeline for Logistic Regression with Bag of Words
pipeline_bow_lr = Pipeline([
    ('vectorizer', CountVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Perform hyper-parameter optimization with GridSearchCV
param_grid_lr = {
    'classifier__penalty': ['l2'],
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['liblinear', 'saga']
}

# Perform grid search with cross-validation
grid_bow_lr = GridSearchCV(pipeline_bow_lr, param_grid=param_grid_lr, cv=3, scoring='accuracy')
grid_bow_lr.fit(X_train_bow_tfidf, y_train_bow_tfidf)

# Best hyper-parameters and cross-validation score
print("Best Hyper-parameters for Logistic Regression with Bag of Words:", grid_bow_lr.best_params_)
print("Best Cross-validation Accuracy for Logistic Regression with Bag of Words:", grid_bow_lr.best_score_)

# Evaluate the best model on the test set
y_pred_bow_lr = grid_bow_lr.predict(X_test_bow_tfidf)
lr_bow_test_accuracy = accuracy_score(y_test_bow_tfidf, y_pred_bow_lr)
print("Test Accuracy:", nb_bow_test_accuracy)

Best Hyper-parameters for Logistic Regression with Bag of Words: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best Cross-validation Accuracy for Logistic Regression with Bag of Words: 0.5837364322859814
Test Accuracy: 0.5861003861003861


### Logistic Regression with TF-IDF

In [48]:
# Create a pipeline for Logistic Regression with TF-IDF
pipeline_tfidf_lr = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Perform hyper-parameter optimization with GridSearchCV
param_grid_lr = {
    'classifier__penalty': ['l2'],
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['liblinear', 'saga']
}

# Perform grid search with cross-validation
grid_tfidf_lr = GridSearchCV(pipeline_tfidf_lr, param_grid=param_grid_lr, cv=3, scoring='accuracy')
grid_tfidf_lr.fit(X_train_bow_tfidf, y_train_bow_tfidf)

# Best hyper-parameters and cross-validation score
print("Best Hyper-parameters for Logistic Regression with TF-IDF:", grid_tfidf_lr.best_params_)
print("Best Cross-validation Accuracy for Logistic Regression with TF-IDF:", grid_tfidf_lr.best_score_)

# Evaluate the best model on the test set
y_pred_tfidf_lr = grid_tfidf_lr.predict(X_test_bow_tfidf)
lr_tfidf_test_accuracy = accuracy_score(y_test_bow_tfidf, y_pred_tfidf_lr)
print("Test Accuracy:", lr_tfidf_test_accuracy)

Best Hyper-parameters for Logistic Regression with TF-IDF: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best Cross-validation Accuracy for Logistic Regression with TF-IDF: 0.5868284436383585
Test Accuracy: 0.5861003861003861


## Embedding-based Model

### LSTM with Different Embeddings

#### Load Pre-trained Embeddings

In [49]:
import gensim
import gensim.downloader as api

# Define paths to save/load models
model_base_path = os.path.join(base_path, 'embedding_models')
os.makedirs(model_base_path, exist_ok=True)

word2vec_path = os.path.join(model_base_path, 'word2vec-google-news-300')
glove_path = os.path.join(model_base_path, 'glove-wiki-gigaword-300')
fasttext_path = os.path.join(model_base_path, 'fasttext-wiki-news-subwords-300')

# Function to download and save models if not already present
def load_or_download_model(model_name, model_path, download_func):
    if os.path.exists(model_path):
        print(f"Loading {model_name} model from Google Drive...")
        model = gensim.models.KeyedVectors.load(model_path)
    else:
        print(f"Downloading {model_name} model...")
        model = download_func()
        model.save(model_path)
        print(f"{model_name} model saved to Google Drive.")
    return model

# Load or download models
word2vec_model = load_or_download_model('Word2Vec', word2vec_path, lambda: api.load('word2vec-google-news-300'))
glove_model = load_or_download_model('GloVe', glove_path, lambda: api.load('glove-wiki-gigaword-300'))
fasttext_model = load_or_download_model('FastText', fasttext_path, lambda: api.load('fasttext-wiki-news-subwords-300'))

# Now, you can create the embedding matrices as before
def create_embedding_matrix(word_index, embedding_model, embedding_dim):
    embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
    for word, i in word_index.items():
        if word in embedding_model:
            embedding_matrix[i] = embedding_model[word]
    return embedding_matrix

# Assuming tokenizer and word_index are already defined
embedding_dim_word2vec = 300  # Word2Vec GoogleNews vectors have 300 dimensions
embedding_dim_glove = 300  # GloVe vectors have 300 dimensions
embedding_dim_fasttext = 300  # FastText vectors have 300 dimensions

embedding_matrix_word2vec = create_embedding_matrix(tokenizer.word_index, word2vec_model, embedding_dim_word2vec)
embedding_matrix_glove = create_embedding_matrix(tokenizer.word_index, glove_model, embedding_dim_glove)
embedding_matrix_fasttext = create_embedding_matrix(tokenizer.word_index, fasttext_model, embedding_dim_fasttext)

print(f"Word2Vec embedding matrix shape: {embedding_matrix_word2vec.shape}")
print(f"GloVe embedding matrix shape: {embedding_matrix_glove.shape}")
print(f"FastText embedding matrix shape: {embedding_matrix_fasttext.shape}")

Loading Word2Vec model from Google Drive...
Loading GloVe model from Google Drive...
Loading FastText model from Google Drive...
Word2Vec embedding matrix shape: (14191, 300)
GloVe embedding matrix shape: (14191, 300)
FastText embedding matrix shape: (14191, 300)


### Manually Perform Hyperparameter Tuning for LSTM

#### Define Hyperparameters and Perform Cross-Validation

In [50]:
import json

# Define paths to save/load results
results_base_path = os.path.join(base_path, 'model_results')
os.makedirs(results_base_path, exist_ok=True)

word2vec_results_path = os.path.join(results_base_path, 'word2vec_results.json')
glove_results_path = os.path.join(results_base_path, 'glove_results.json')
fasttext_results_path = os.path.join(results_base_path, 'fasttext_results.json')

# Helper function to check if results exist
def load_results(results_path):
    if os.path.exists(results_path):
        with open(results_path, 'r') as file:
            results = json.load(file)
        return results['best_accuracy'], results['best_params']
    return None, None

# Helper function to save results
def save_results(results_path, best_accuracy, best_params):
    with open(results_path, 'w') as file:
        json.dump({'best_accuracy': best_accuracy, 'best_params': best_params}, file)

In [65]:
from sklearn.model_selection import KFold
from tensorflow.keras.callbacks import EarlyStopping

# Function to build simplified LSTM model
def build_simplified_lstm_model(embedding_matrix, embedding_dim, units, dropout_rate, recurrent_dropout_rate):
    model = Sequential()
    model.add(Embedding(input_dim=embedding_matrix.shape[0],
                        output_dim=embedding_dim,
                        weights=[embedding_matrix],
                        input_length=200,
                        trainable=False))
    model.add(LSTM(units, dropout=dropout_rate, recurrent_dropout=recurrent_dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Define hyperparameters to tune
units = 32  # Fixed number of LSTM units
dropout_rate_list = [0.2, 0.3, 0.4]
recurrent_dropout_rate_list = [0.2, 0.3, 0.4]
batch_size = 256  # Fixed batch size
epochs = 100  # Reduce the number of epochs for faster training
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Perform cross-validation and hyperparameter tuning manually
def manual_hyperparameter_tuning(X, y, embedding_matrix, embedding_dim):
    best_accuracy = 0
    best_params = {}
    kf = KFold(n_splits=3, shuffle=True, random_state=42)

    for dropout_rate in dropout_rate_list:
        for recurrent_dropout_rate in recurrent_dropout_rate_list:
            accuracies = []
            for train_index, val_index in kf.split(X):
                X_train, X_val = X[train_index], X[val_index]
                y_train, y_val = y[train_index], y[val_index]

                model = build_simplified_lstm_model(embedding_matrix, embedding_dim, units, dropout_rate, recurrent_dropout_rate)
                model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=0)

                y_pred = (model.predict(X_val) > 0.5).astype("int32")
                accuracies.append(accuracy_score(y_val, y_pred))

            avg_accuracy = np.mean(accuracies)
            if avg_accuracy > best_accuracy:
                best_accuracy = avg_accuracy
                best_params = {
                    'dropout_rate': dropout_rate,
                    'recurrent_dropout_rate': recurrent_dropout_rate
                }
            print(f"Params: {dropout_rate}, {recurrent_dropout_rate} - Accuracy: {avg_accuracy}")
    return best_accuracy, best_params

In [52]:
# Function to perform manual hyperparameter tuning with option to retrain
def tune_hyperparameters(X, y, embedding_matrix, embedding_dim, results_path, retrain=False):
    best_accuracy, best_params = load_results(results_path)

    if not retrain and best_accuracy is not None and best_params is not None:
        print(f"Loaded existing results from {results_path}")
        return best_accuracy, best_params

    print("Performing hyperparameter tuning...")
    best_accuracy, best_params = manual_hyperparameter_tuning(X, y, embedding_matrix, embedding_dim)
    save_results(results_path, best_accuracy, best_params)
    print(f"Saved results to {results_path}")

    return best_accuracy, best_params

In [53]:
# Tune hyperparameters for Word2Vec embeddings
print("Tuning hyperparameters for Word2Vec embeddings")
best_accuracy_word2vec, best_params_word2vec = tune_hyperparameters(X_train_pad, y_train, embedding_matrix_word2vec, embedding_dim_word2vec, word2vec_results_path, retrain=False)
print("Best accuracy for Word2Vec: ", best_accuracy_word2vec)
print("Best params for Word2Vec: ", best_params_word2vec)

Tuning hyperparameters for Word2Vec embeddings
Loaded existing results from content/drive/MyDrive/model_results/word2vec_results.json
Best accuracy for Word2Vec:  0.6150253301313807
Best params for Word2Vec:  {'dropout_rate': 0.3, 'recurrent_dropout_rate': 0.3, 'batch_size': 256, 'epochs': 100}


In [54]:
# Tune hyperparameters for GloVe embeddings
print("Tuning hyperparameters for GloVe embeddings")
best_accuracy_glove, best_params_glove = tune_hyperparameters(X_train_pad, y_train, embedding_matrix_glove, embedding_dim_glove, glove_results_path, retrain=False)
print("Best accuracy for GloVe: ", best_accuracy_glove)
print("Best params for GloVe: ", best_params_glove)

Tuning hyperparameters for GloVe embeddings
Loaded existing results from content/drive/MyDrive/model_results/glove_results.json
Best accuracy for GloVe:  0.6123261737227176
Best params for GloVe:  {'dropout_rate': 0.4, 'recurrent_dropout_rate': 0.3, 'batch_size': 256, 'epochs': 100}


In [55]:
# Tune hyperparameters for FastText embeddings
print("Tuning hyperparameters for FastText embeddings")
best_accuracy_fasttext, best_params_fasttext = tune_hyperparameters(X_train_pad, y_train, embedding_matrix_fasttext, embedding_dim_fasttext, fasttext_results_path, retrain=False)
print("Best accuracy for FastText: ", best_accuracy_fasttext)
print("Best params for FastText: ", best_params_fasttext)

Tuning hyperparameters for FastText embeddings
Loaded existing results from content/drive/MyDrive/model_results/fasttext_results.json
Best accuracy for FastText:  0.6028563431687015
Best params for FastText:  {'dropout_rate': 0.4, 'recurrent_dropout_rate': 0.2, 'batch_size': 256, 'epochs': 100}


### Evaluate on Test Data

#### Evaluate LSTM Models on Test Data

In [66]:
# Function to evaluate the model on test data
def evaluate_lstm_model(X_train, y_train, X_test, y_test, embedding_matrix, embedding_dim, best_params):
    model = build_simplified_lstm_model(embedding_matrix, embedding_dim,
                                        units=units,  # Fixed number of LSTM units
                                        dropout_rate=best_params['dropout_rate'],
                                        recurrent_dropout_rate=best_params['recurrent_dropout_rate'])
    model.fit(X_train, y_train,
              epochs=epochs,
              batch_size=batch_size,
              validation_split=0.1,
              callbacks=[early_stopping],
              verbose=0)

    y_pred = (model.predict(X_test) > 0.5).astype("int32")
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

In [67]:
# Evaluate the models on test data
print("Evaluating Word2Vec model on test data")
accuracy_word2vec_test = evaluate_lstm_model(X_train_pad, y_train, X_test_pad, y_test, embedding_matrix_word2vec, embedding_dim_word2vec, best_params_word2vec)
print("Test Accuracy for Word2Vec: ", accuracy_word2vec_test)

Evaluating Word2Vec model on test data
41/41 [==============================] - 3s 67ms/step
Test Accuracy for Word2Vec:  0.6030888030888031


In [58]:
print("Evaluating GloVe model on test data")
accuracy_glove_test = evaluate_lstm_model(X_train_pad, y_train, X_test_pad, y_test, embedding_matrix_glove, embedding_dim_glove, best_params_glove)
print("Test Accuracy for GloVe: ", accuracy_glove_test)

Evaluating GloVe model on test data
41/41 [==============================] - 3s 64ms/step
Test Accuracy for GloVe:  0.6131274131274131


In [59]:
print("Evaluating FastText model on test data")
accuracy_fasttext_test = evaluate_lstm_model(X_train_pad, y_train, X_test_pad, y_test, embedding_matrix_fasttext, embedding_dim_fasttext, best_params_fasttext)
print("Test Accuracy for FastText: ", accuracy_fasttext_test)

Evaluating FastText model on test data
41/41 [==============================] - 3s 59ms/step
Test Accuracy for FastText:  0.6054054054054054


## Summarise and Display Results

#### Function to Summarize Results
We'll create a function to collect the results and another function to display them in a DataFrame.

In [60]:
# Function to add results
def add_results(model_type, features, parameters, val_accuracy, test_accuracy):
    results.append({
        'Model': model_type,
        'Features': features,
        'Hyper-Parameters': parameters,
        'Validation Accuracy': val_accuracy,
        'Test Accuracy': test_accuracy
    })

# Function to display results
def display_results():
    df_results = pd.DataFrame(results)
    return df_results

#### Add Results from Each Evaluation
After evaluating each model, add the results to the list.

In [68]:
# List to hold results
results = []

# Naive Bayes with Bag of Words
add_results('Naive Bayes', 'Bag of Word', grid_bow_nb.best_params_, grid_bow_nb.best_score_, nb_bow_test_accuracy)
# Naive Bayes with TF-IDF
add_results('Naive Bayes', 'TF-IDF', grid_tfidf_nb.best_params_, grid_tfidf_nb.best_score_, accuracy_glove_test)

# Logistic Regression with Bag of Words
add_results('Logistic Regression', 'Bag of Word', grid_bow_lr.best_params_, grid_bow_lr.best_score_, accuracy_word2vec_test)
# Logistic Regression with TF-IDF
add_results('Logistic Regression', 'TF-IDF', grid_tfidf_lr.best_params_, grid_tfidf_lr.best_score_, accuracy_glove_test)

# LSTM with Word2Vec
add_results('LSTM', 'Word2Vec', best_params_word2vec, best_accuracy_word2vec, accuracy_word2vec_test)
# LSTM with Glove
add_results('LSTM', 'GloVe', best_params_glove, best_accuracy_glove, accuracy_glove_test)
# LSTM with FastTest
add_results('LSTM', 'FastText', best_params_fasttext, best_accuracy_fasttext, accuracy_fasttext_test)

#### Display Results
Display the results in a DataFrame.

In [69]:
# Function to display results
def display_results():
    df_results = pd.DataFrame(results)
    return df_results

# Display the results in a DataFrame
df_results = display_results()
df_results

,Model,Features,Hyper-Parameters,Validation Accuracy,Test Accuracy
0,Naive Bayes,Bag of Word,{'classifier__alpha': 1.0},0.589532,0.586100
1,Naive Bayes,TF-IDF,{'classifier__alpha': 2.0},0.591464,0.613127
2,Logistic Regression,Bag of Word,"{'classifier__C': 1, 'classifier__penalty': 'l...",0.583736,0.603089
3,Logistic Regression,TF-IDF,"{'classifier__C': 1, 'classifier__penalty': 'l...",0.586828,0.613127
4,LSTM,Word2Vec,"{'dropout_rate': 0.3, 'recurrent_dropout_rate'...",0.615025,0.603089
5,LSTM,GloVe,"{'dropout_rate': 0.4, 'recurrent_dropout_rate'...",0.612326,0.613127
6,LSTM,FastText,"{'dropout_rate': 0.4, 'recurrent_dropout_rate'...",0.602856,0.605405
